In [13]:
import pandas as pd

In [14]:
stocks_raw_data = pd.read_parquet("nifty_500_stocks_data.parquet")
display(stocks_raw_data.head())

,Date,Ticker,Open,High,Low,Close,Volume
0,2019-09-19,360ONE,244.034802,256.236542,244.034802,256.236542,7109860
1,2019-09-20,360ONE,268.034927,269.043335,268.034927,269.043335,2760648
2,2019-09-23,360ONE,282.495483,282.495483,282.495483,282.495483,34096
3,2019-09-24,360ONE,296.613220,296.613220,296.613220,296.613220,2496652
4,2019-09-25,360ONE,311.436801,311.436801,281.789598,283.352631,530580


In [19]:
def rolling_average(df, window=50):
    """
    Calculate the rolling average for each stock in the DataFrame.

    Parameters:
    df (pd.DataFrame): DataFrame containing stock data with 'Date' and 'Close' columns.
    window (int): The window size for calculating the rolling average.

    Returns:
    pd.DataFrame: DataFrame with an additional column for the rolling average.
    """
    df = df.copy()
    df[f'Rolling_Avg_{window}'] = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=window).mean())
    return df

def trending_stocks(df, rolling_average_column):
    """
    A stock can satisfy the first two conditions while the trend is fading. 
    After a long advance, the price may stall, the 50DMA flattens and begins to curl over, and yet “Close > 50DMA > 200DMA” is still true for weeks. 
    The slope condition catches this.
    In  code today’s 50DMA is compared with the value 10 days ago, so it asks “is the average cost of recent buyers higher than two weeks ago?


    Parameters:
    df (pd.DataFrame): DataFrame containing stock data with 'Date' and 'Close' columns.
    window (int): The window size for calculating the rolling average.

    Returns:
    pd.DataFrame: DataFrame with an additional column indicating if the stock is trending.
    """
    df['Trending'] = df.groupby("Ticker")[rolling_average_column].diff(10) > 0
    return df
stocks_with_rolling_avg = rolling_average(stocks_raw_data, window=50)
stocks_with_rolling_avg = rolling_average(stocks_with_rolling_avg, window=200)
stocks_with_rolling_avg = trending_stocks(stocks_with_rolling_avg, 'Rolling_Avg_50')
stocks_with_rolling_avg

,Date,Ticker,Open,High,Low,Close,Volume,Rolling_Avg_50,Rolling_Avg_200,Trending
0,2019-09-19,360ONE,244.034802,256.236542,244.034802,256.236542,7109860,NaN,NaN,False
1,2019-09-20,360ONE,268.034927,269.043335,268.034927,269.043335,2760648,NaN,NaN,False
2,2019-09-23,360ONE,282.495483,282.495483,282.495483,282.495483,34096,NaN,NaN,False
3,2019-09-24,360ONE,296.613220,296.613220,296.613220,296.613220,2496652,NaN,NaN,False
4,2019-09-25,360ONE,311.436801,311.436801,281.789598,283.352631,530580,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...
1932970,2026-11-03,ECLERX,1598.599976,1598.599976,1553.800049,1578.000000,143095,1881.835994,1734.960969,False
1932971,2026-11-04,ECLERX,1584.000000,1598.400024,1536.099976,1562.300049,180734,1876.767996,1731.287836,False
1932972,2026-11-05,ECLERX,1583.900024,1619.199951,1576.500000,1588.500000,185349,1872.081995,1727.751199,False
1932973,2026-11-06,ECLERX,1586.900024,1622.900024,1562.099976,1600.599976,156092,1866.503994,1724.254075,False


In [23]:
import numpy as np
import pandas as pd

def add_adx(df: pd.DataFrame, n: int = 14) -> pd.DataFrame:
    df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    g = df.groupby("Ticker", sort=False)

    # 1. Directional movement and true range
    up = g["High"].diff()
    down = -g["Low"].diff()
    prev_close = g["Close"].shift()

    df["plus_dm"] = up.where((up > down) & (up > 0), 0.0)
    df["minus_dm"] = down.where((down > up) & (down > 0), 0.0)
    df["tr"] = pd.concat(
        [df.High - df.Low,
         (df.High - prev_close).abs(),
         (df.Low - prev_close).abs()], axis=1
    ).max(axis=1)

    # 2. Wilder smoothing (an EMA with alpha = 1/n), per ticker
    sm = (df.groupby("Ticker")[["plus_dm", "minus_dm", "tr"]]
            .ewm(alpha=1 / n, adjust=False, min_periods=n).mean()
            .reset_index(level=0, drop=True))

    # 3. Directional indicators and DX
    df[f"plus_di"] = 100 * sm["plus_dm"] / sm["tr"]
    df[f"minus_di"] = 100 * sm["minus_dm"] / sm["tr"]
    di_sum = (df.plus_di + df.minus_di).replace(0, np.nan)
    dx = 100 * (df.plus_di - df.minus_di).abs() / di_sum

    # 4. ADX = Wilder-smoothed DX, per ticker
    df["ADX"] = (dx.groupby(df["Ticker"])
                   .ewm(alpha=1 / n, adjust=False, min_periods=n).mean()
                   .reset_index(level=0, drop=True))

    return df.drop(columns=["plus_dm", "minus_dm", "tr"])

stocks_with_rolling_avg = add_adx(stocks_with_rolling_avg)
display(stocks_with_rolling_avg.head())

,Date,Ticker,Open,High,Low,Close,Volume,Rolling_Avg_50,Rolling_Avg_200,Trending,plus_di,minus_di,ADX
0,2019-09-19,360ONE,244.034802,256.236542,244.034802,256.236542,7109860,NaN,NaN,False,NaN,NaN,NaN
1,2019-09-20,360ONE,268.034927,269.043335,268.034927,269.043335,2760648,NaN,NaN,False,NaN,NaN,NaN
2,2019-09-23,360ONE,282.495483,282.495483,282.495483,282.495483,34096,NaN,NaN,False,NaN,NaN,NaN
3,2019-09-24,360ONE,296.613220,296.613220,296.613220,296.613220,2496652,NaN,NaN,False,NaN,NaN,NaN
4,2019-09-25,360ONE,311.436801,311.436801,281.789598,283.352631,530580,NaN,NaN,False,NaN,NaN,NaN


In [24]:
def add_efficiency_ratio(df: pd.DataFrame, n: int = 60) -> pd.DataFrame:
    df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    g = df.groupby("Ticker", sort=False)["Close"]

    # 1. Net move: today's close minus the close n days ago, per ticker
    net = g.diff(n)

    # 2. Path: sum of absolute daily changes over the same n days
    daily_move = g.diff().abs()
    path = daily_move.groupby(df["Ticker"], sort=False).transform(
        lambda s: s.rolling(n, min_periods=n).sum()
    )

    # 3. Ratio (guard against a zero path, e.g. a stock that never moved)
    path = path.replace(0, np.nan)
    df[f"ER{n}"] = net.abs() / path          # smoothness only
    df[f"ER{n}_signed"] = net / path         # smoothness plus direction
    return df

In [25]:
stocks_with_rolling_avg = add_efficiency_ratio(stocks_with_rolling_avg)
stocks_with_rolling_avg

,Date,Ticker,Open,High,Low,Close,Volume,Rolling_Avg_50,Rolling_Avg_200,Trending,plus_di,minus_di,ADX,ER60,ER60_signed
0,2019-09-19,360ONE,244.034802,256.236542,244.034802,256.236542,7109860,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN
1,2019-09-20,360ONE,268.034927,269.043335,268.034927,269.043335,2760648,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN
2,2019-09-23,360ONE,282.495483,282.495483,282.495483,282.495483,34096,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN
3,2019-09-24,360ONE,296.613220,296.613220,296.613220,296.613220,2496652,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN
4,2019-09-25,360ONE,311.436801,311.436801,281.789598,283.352631,530580,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1932970,2026-11-03,ZYDUSWELL,485.000000,526.849976,466.799988,493.350006,3629654,513.669001,491.038446,False,23.444898,15.708017,27.111172,0.068906,-0.068906
1932971,2026-11-04,ZYDUSWELL,499.899994,499.899994,485.299988,488.799988,142028,513.629001,491.280345,False,22.320420,14.954620,26.586137,0.081047,-0.081047
1932972,2026-11-05,ZYDUSWELL,488.899994,505.850006,469.799988,474.700012,350122,512.901002,491.450497,False,19.795715,18.126414,25.001550,0.074973,-0.074973
1932973,2026-11-06,ZYDUSWELL,475.000000,479.649994,465.100006,468.600006,136631,511.764001,491.578921,False,18.868078,18.790703,23.230401,0.068393,-0.068393


In [26]:
stocks_with_rolling_avg.ER60.describe()

count    1.900878e+06
mean     1.488394e-01
std      1.113989e-01
min      0.000000e+00
25%      6.053966e-02
50%      1.273781e-01
75%      2.145590e-01
max      1.000000e+00
Name: ER60, dtype: float64

In [27]:
stocks_with_rolling_avg.ER60.quantile([.5, .8, .9, .95])

0.50    0.127378
0.80    0.238165
0.90    0.303073
0.95    0.358815
Name: ER60, dtype: float64

In [29]:
up = stocks_with_rolling_avg[stocks_with_rolling_avg.Trending & stocks_with_rolling_avg.ER60_signed.gt(0)]      # trending up and net move is positive
print(up.ER60.describe())
print(up.ER60.quantile([.25, .5, .75, .9, .95]))

count    1.006868e+06
mean     1.725770e-01
std      1.136757e-01
min      7.243456e-08
25%      8.448470e-02
50%      1.549576e-01
75%      2.418211e-01
max      1.000000e+00
Name: ER60, dtype: float64
0.25    0.084485
0.50    0.154958
0.75    0.241821
0.90    0.327881
0.95    0.381840
Name: ER60, dtype: float64


In [36]:
latest = stocks_with_rolling_avg[stocks_with_rolling_avg.Date == "2026-10-09"]
print(len(latest), "stocks on latest date")
print((latest.Trending).sum(), "pass step 1")
print((latest.Trending & (latest.ADX > 25)).sum(), "pass steps 1 + 2")
print((latest.Trending & (latest.ADX > 25) & (latest.ER60 > 0.25)).sum(), "pass steps 1 + 2 + 3")

497 stocks on latest date
146 pass step 1
55 pass steps 1 + 2
10 pass steps 1 + 2 + 3


In [34]:
stocks_with_rolling_avg.Date.max()

Timestamp('2026-11-09 00:00:00')